# M3 — Agente con tool use (Ola 3) — validación en Colab
### SI4006 · Proyecto Integrador · Universidad EAFIT

Este notebook corre `scripts/agente_tools.py` con GPU. El agente carga **tres modelos a
la vez** (el cerebro Qwen2.5-3B-Instruct, el generador del RAG Qwen2.5-1.5B-Instruct, y
BETO+LoRA de M1) — eso no cabe cómodo en una laptop sin GPU, por eso esto se corre aquí.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU
(T4)`.

Qué hace este notebook, en orden:
1. Clona el repo (rama `juanes`) e instala dependencias.
2. Corre los tests de lógica pura del agente (parseo de decisiones) — sin GPU.
3. Prueba cada herramienta por separado (BETO, luego el RAG avanzado).
4. Corre el agente completo (ciclo pensar → actuar → observar) sobre preguntas de
   ejemplo, incluida una fuera de dominio para probar que el agente sabe no forzar
   una respuesta.

## 1 · Preparar el entorno

In [ ]:
import os

# Si esta VM de Colab ya tenía el repo clonado de una corrida anterior, "git clone"
# fallaría en silencio (la carpeta ya existe) y seguiríamos con código VIEJO sin
# darnos cuenta -- por eso, si ya existe, se hace git pull en vez de clonar de cero.
if os.path.isdir("triage-dermatologico-ia"):
    %cd triage-dermatologico-ia
    !git checkout juanes
    !git pull origin juanes
else:
    !git clone --branch juanes https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd triage-dermatologico-ia

!git log -1 --oneline  # para confirmar a simple vista qué commit quedó activo

In [ ]:
%pip install -q transformers datasets peft accelerate pandas torch scikit-learn sentence-transformers chromadb rank-bm25 pyyaml

# Mismo arreglo que en M2_juez_colab.ipynb: Colab trae una version vieja de
# torchao preinstalada que hace tronar a `peft` al importarse (ImportError),
# aunque este proyecto no cuantiza con torchao para nada.
!pip uninstall -y -q torchao

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU activa. Ve a Entorno de ejecución -> Cambiar tipo de "
          "entorno de ejecución -> T4 GPU, y vuelve a correr esta celda.")

## 2 · Tests de lógica pura del agente (sin modelo, sin GPU)

Ya se corrieron en local durante el desarrollo; se repiten aquí para dejar registro de
que el código que se va a ejecutar con los modelos reales pasa estas pruebas: parseo de
decisiones con JSON limpio, con texto extra alrededor, herramienta inexistente, sin
JSON, JSON roto, acción desconocida, y que `_armar_resultado` combina contexto/fuentes
sin duplicados.

In [ ]:
import sys
sys.path.insert(0, "scripts")
import agente_tools as at

casos = [
    ('{"accion": "usar_herramienta", "herramienta": "clasificador_beto"}',
     {"accion": "usar_herramienta", "herramienta": "clasificador_beto"}),
    ('{"accion": "responder_final", "respuesta": "Esto no es urgente."}',
     {"accion": "responder_final", "respuesta": "Esto no es urgente."}),
    ('Claro, aqui esta: {"accion": "usar_herramienta", "herramienta": "buscar_informacion_dermatologica"} listo',
     {"accion": "usar_herramienta", "herramienta": "buscar_informacion_dermatologica"}),
]
for texto, esperado in casos:
    r = at._parsear_decision(texto)
    assert r == esperado, r

for texto_malo in [
    '{"accion": "usar_herramienta", "herramienta": "buscar_en_google"}',
    'no se que hacer con esta pregunta',
    '{"accion": "usar_herramienta", "herramienta": }',
    '{"accion": "pensar_en_voz_alta"}',
]:
    r = at._parsear_decision(texto_malo)
    assert r["accion"] == "error", r

resultados = {
    "clasificador_beto": {"herramienta": "clasificador_beto", "etiqueta": "urgente"},
    "buscar_informacion_dermatologica": {"herramienta": "buscar_informacion_dermatologica",
        "contexto": ["texto A", "texto B"], "fuentes": ["doc1", "doc2", "doc1"]},
}
r = at._armar_resultado("respuesta final", resultados, ["clasificador_beto", "buscar_informacion_dermatologica"])
assert r["fuentes"] == ["doc1", "doc2"], r["fuentes"]
assert r["contexto"] == ["texto A", "texto B"], r["contexto"]

print("Todos los tests de lógica pura pasaron.")

## 3 · Probar cada herramienta por separado

Antes de dejar que el agente decida solo, confirmamos que las dos piezas que va a usar
funcionan bien de forma aislada.

In [ ]:
r_beto = at.tool_clasificador_beto("Tengo un lunar que me cambió de color y de forma en los últimos meses.")
print("BETO:", r_beto)

In [ ]:
r_rag = at.tool_buscar_informacion_dermatologica("Tengo un lunar que me cambió de color y de forma en los últimos meses.")
print("RAG - etiqueta:", r_rag["etiqueta"])
print("RAG - fuentes:", r_rag["fuentes"])
print("RAG - respuesta:", r_rag["respuesta"][:300])

## 4 · El agente completo — ciclo pensar → actuar → observar

Tres preguntas de prueba: una claramente urgente (para ver si encadena las dos
herramientas), una claramente benigna, y una que no es sobre piel en absoluto (para
comprobar que el agente sabe reconocer cuándo ninguna herramienta aplica en vez de
forzar una respuesta).

In [ ]:
preguntas_de_prueba = [
    "Tengo un lunar que me cambió de color y de forma en los últimos meses.",
    "Me salió un bultico firme bajo la piel, no crece ni duele.",
    "Tengo dolor de cabeza fuerte y fiebre desde ayer, ¿debería preocuparme?",
]

resultados_agente = []
for pregunta in preguntas_de_prueba:
    print(f"\n=== Pregunta: {pregunta} ===")
    resultado = at.agente(pregunta)
    print(f"Tools usadas: {resultado['tools_usadas']}")
    print(f"Respuesta: {resultado['respuesta'][:300]}")
    print(f"Fuentes: {resultado['fuentes']}")
    resultados_agente.append({"pregunta": pregunta, **resultado})

## 5 · Guardar resultados

Para dejar evidencia y que Camilo (Ola 4) pueda ver ejemplos reales del agente
funcionando al armar su evaluación final.

In [ ]:
import json as _json

with open("resultados_agente_m3.json", "w", encoding="utf-8") as f:
    _json.dump(resultados_agente, f, ensure_ascii=False, indent=2)
print("Guardado: resultados_agente_m3.json")

In [ ]:
from google.colab import files
files.download("resultados_agente_m3.json")

---

Con esto queda validado el agente con el modelo y las herramientas reales: las dos
herramientas funcionan por separado, el ciclo pensar/actuar/observar encadena
`clasificador_beto` y `buscar_informacion_dermatologica` cuando hace falta, y reconoce
cuándo una pregunta está fuera de su alcance sin forzar ninguna herramienta.